In [1]:
import math
import pandas as pd

from sampling_manager import SamplingManager
from latents_generator import latents_generator

In [2]:
# Start by loading our data
df = pd.read_csv('./data/train_20k.csv')
# df = pd.read_csv('./data/train_50k.csv')

# Testing
def print_column_distributions(df, columns):
    for col in columns:
        print(f"\nDistribution for '{col}':")
        print(df[col].value_counts(dropna=False, normalize=True))

In [3]:
criteria = {
    # "movecount": { (0, 9): .1, (10, 19): .3, (20, 29): .3,
    #                (30, 39): .2, (40, None): .1 },
    # "player":    { "w": .5, "b": .5 }
}

sm = SamplingManager(df, criteria)

# Generate our Data

In [4]:
# =========================
# IS_CHECK
# =========================
IS_CHECK_CONFIG = {   
    "tp": 0.8    # Determines % of checks that are 'Yes' (correct color is asked)
}

new_df = latents_generator("is_check", sm.df, IS_CHECK_CONFIG)
# new_df.head()

sm.df = new_df
samples = sm.get_samples(
    500,
    criteria={
        **criteria,
        "movecount": { (0, 9): .1, (10, 19): .3, (20, 29): .3,
                    (30, 39): .2, (40, None): .1 },
        "player":    { "w": .5, "b": .5 },
        "is_check": {"n": .5, "w": .25, "b": .25}, 
        "is_check_gen": {"tp": .4, "fp": .1, "tn": .5}, 
    }
)

cols_to_check = ["movecount_bucket", "player_bucket", "is_check_bucket", "is_check_gen_bucket"]
print_column_distributions(samples, cols_to_check)


Distribution for 'movecount_bucket':
movecount_bucket
10-19    0.304
20-29    0.302
30-39    0.200
0-9      0.098
40+      0.096
Name: proportion, dtype: float64

Distribution for 'player_bucket':
player_bucket
w    0.5
b    0.5
Name: proportion, dtype: float64

Distribution for 'is_check_bucket':
is_check_bucket
n    0.50
w    0.25
b    0.25
Name: proportion, dtype: float64

Distribution for 'is_check_gen_bucket':
is_check_gen_bucket
tn    0.5
tp    0.4
fp    0.1
Name: proportion, dtype: float64


In [58]:
new_df = samples.sample(frac=1).reset_index(drop=True)

for _, row in new_df.iterrows():
    sys, usr, ast = row['is_check_chat']['chat']
    print(f"Info || is_check_bucket: {row['is_check_bucket']} | is_check_gen_cat: {row['is_check_gen_bucket']}")    # IS_CHECK
    print(f"System: {sys[1]} ")
    print(f"User:\n{usr[1]}\n")
    print(f"Assistant:\n{ast[1]}\n")
    break

Info || is_check_bucket: w | is_check_gen_cat: fp
System: chess_generic.txt 
User:
Here is a board in a game you're currnetly playing:
8| . . . . . . . .
7| . . . . . . . .
6| . . . . . . . .
5| . . . . . . . .
4| . . . . . k . .
3| . . . . . p p p
2| . . . . . K . .
1| . . . . . . . .
   _ _ _ _ _ _ _ _
   A B C D E F G H

- It is White’s turn to move.
- No castling rights available.
- No en passant target square.
- Halfmove clock: 0
- Fullmove number: 52

I want you to respond immediately with a single token -- 'Yes' or 'No' -- to answer my desired question:

Is the black king in check?

Assistant:
No

